In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pypfopt import risk_models, expected_returns
from pypfopt.black_litterman import BlackLittermanModel
from pypfopt.black_litterman import market_implied_prior_returns
from pypfopt.efficient_frontier import EfficientFrontier

### Lấy dữ liệu và tính toán thông số thị trường

In [ ]:
import yfinance as yf

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
data = yf.download(tickers, start="2020-01-01", end="2023-01-01")['Close']
returns = data.pct_change().dropna()

# Tính lợi nhuận kỳ vọng theo mô hình thị trường (CAPM)
mu_market = expected_returns.capm_return(data)
S = risk_models.sample_cov(data)

### Xây dựng quan điểm nhà đầu tư

In [3]:
# Nhà đầu tư tin rằng Apple sẽ có lợi nhuận cao hơn Microsoft 2%
views = {
    "AAPL": 0.02,  # 2% lợi nhuận kỳ vọng cao hơn giá trị thị trường
    "MSFT": -0.01  # Microsoft bị định giá cao, sẽ giảm 1%
}

# Mức độ tin cậy của quan điểm
q = np.array([0.02, -0.01])  # Lợi nhuận kỳ vọng của Apple, Microsoft
p = np.array([[1, 0, 0, 0],  # Chỉ tác động đến Apple
              [0, 1, 0, 0]])  # Chỉ tác động đến Microsoft
omega = np.diag([0.0001, 0.0001])  # Độ không chắc chắn của quan điểm

### Kết hợp vào mô hình Black-Litterman

In [4]:
bl = BlackLittermanModel(S, pi=mu_market, P=p, Q=q, omega=omega)
mu_bl = bl.bl_returns()  # Lợi nhuận kỳ vọng điều chỉnh theo Black-Litterman

### Tối ưu danh mục với lợi nhuận mới

In [ ]:
ef = EfficientFrontier(mu_bl, S)
weights = ef.max_sharpe()
ef.portfolio_performance(verbose=True)